# W10–W12 Practice — Mobile Robot Autonomy
## EE0849 Introduction to Robotics

In this single notebook you'll wire together the three things a mobile
robot needs to move on its own:

| Part | What you write | Concept |
|------|----------------|---------|
| 1. **Sense**  | `scan(pose)`        | LiDAR ray cast (W10) |
| 2. **Map**    | `update_grid(...)`  | Log-odds occupancy grid (W10) |
| 3. **Plan deliberately** | `astar(...)` | A\* on the grid (W11) |
| 4. **Plan with sampling** | `rrt(...)` | RRT in continuous space (W12) |
| 5. **Drive**  | (already done) | Smoothing + waypoint follow |

We hide the boring plumbing (geometry, plotting, Bresenham, distance
transforms, smoothing) in **one big setup cell** that you run once and
forget. After that, every cell you fill in is one of the *interesting*
mobile-robot ideas.


In [1]:
# Run once on Colab / fresh envs
!pip install numpy matplotlib scipy scikit-image --quiet


## 📦 Setup — run once, then ignore

This cell defines the world, plotting, geometry, and grid plumbing.
You don't need to read it. Skim only if you're curious.


In [2]:
import numpy as np
import matplotlib.pyplot as plt
import heapq
from math import sqrt, hypot, cos, sin, pi
from skimage.draw import line as bresenham_line
from scipy.ndimage import binary_dilation

rng = np.random.default_rng(0)

# ---- World: a 10x10 room with two rectangular obstacles ----------------
def _box(x, y, w, h):
    return [((x, y), (x + w, y)),
            ((x + w, y), (x + w, y + h)),
            ((x + w, y + h), (x, y + h)),
            ((x, y + h), (x, y))]
SEGMENTS = _box(0, 0, 10, 10) + _box(3, 2, 2, 3) + _box(7, 5, 2, 3)

# ---- Geometry: ray <-> segment intersection (closed-form) --------------
def ray_segment_intersection(origin, direction, seg):
    (Ax, Ay), (Bx, By) = seg
    Ox, Oy = origin
    Dx, Dy = direction
    Vx, Vy = Bx - Ax, By - Ay
    den = Dx * (-Vy) - Dy * (-Vx)
    if abs(den) < 1e-9:
        return None
    t = ((Ax - Ox) * (-Vy) - (Ay - Oy) * (-Vx)) / den
    u = (Dx * (Ay - Oy) - Dy * (Ax - Ox)) / den
    if t >= 0 and 0.0 <= u <= 1.0:
        return t
    return None

# ---- Occupancy grid (10x10 m at 0.1 m/cell -> 100x100) -----------------
GRID_RES = 0.1
GRID_M = int(10 / GRID_RES)
GRID_N = int(10 / GRID_RES)

def world_to_grid(x, y):
    return int(np.clip(x / GRID_RES, 0, GRID_M - 1)), \
           int(np.clip(y / GRID_RES, 0, GRID_N - 1))

def cells_along_ray(pose, alpha, r, max_range=10.0):
    """Return (free_cells, hit_cell_or_None) for one ray.

    Free cells are along the ray; hit_cell is the endpoint cell only when
    the ray actually struck something (r < max_range)."""
    x, y, th = pose
    i0, j0 = world_to_grid(x, y)
    ex = x + r * cos(alpha + th)
    ey = y + r * sin(alpha + th)
    ie, je = world_to_grid(ex, ey)
    rr, cc = bresenham_line(i0, j0, ie, je)
    rr = np.clip(rr, 0, GRID_M - 1)
    cc = np.clip(cc, 0, GRID_N - 1)
    free = list(zip(rr[:-1], cc[:-1]))
    hit = (rr[-1], cc[-1]) if r < max_range - 1e-3 else None
    return free, hit

def to_binary_occupancy(grid_log):
    """Threshold log-odds to a binary 0/1 occupancy grid (1 = blocked)."""
    p = 1.0 / (1.0 + np.exp(-grid_log))
    return ((p > 0.5) & (np.abs(grid_log) > 1e-6)).astype(np.uint8)

def inflate(occ, r_cells=3):
    """Grow obstacles by a robot radius (so we can plan as a point)."""
    s = np.ones((2 * r_cells + 1, 2 * r_cells + 1))
    return binary_dilation(occ.astype(bool), structure=s).astype(np.uint8)

# ---- A* helpers --------------------------------------------------------
def neighbors_8(i, j, M, N):
    SQRT2 = sqrt(2)
    for di, dj, c in [(-1, 0, 1), (1, 0, 1), (0, -1, 1), (0, 1, 1),
                      (-1, -1, SQRT2), (-1, 1, SQRT2),
                      (1, -1, SQRT2), (1, 1, SQRT2)]:
        ni, nj = i + di, j + dj
        if 0 <= ni < M and 0 <= nj < N:
            yield ni, nj, c

def octile(a, b):
    di = abs(a[0] - b[0]); dj = abs(a[1] - b[1])
    return min(di, dj) * sqrt(2) + abs(di - dj)

# ---- RRT helpers -------------------------------------------------------
def nearest(nodes, q):
    return min(range(len(nodes)),
               key=lambda i: hypot(nodes[i][0] - q[0], nodes[i][1] - q[1]))

def steer(qfrom, qto, step=0.5):
    dx, dy = qto[0] - qfrom[0], qto[1] - qfrom[1]
    L = hypot(dx, dy)
    return qto if L < step else (qfrom[0] + dx * step / L,
                                 qfrom[1] + dy * step / L)

def collision_free(p1, p2, occ):
    i1, j1 = world_to_grid(*p1); i2, j2 = world_to_grid(*p2)
    rr, cc = bresenham_line(i1, j1, i2, j2)
    rr = np.clip(rr, 0, occ.shape[0] - 1)
    cc = np.clip(cc, 0, occ.shape[1] - 1)
    return not occ[rr, cc].any()

# ---- Smoothing & path metrics -----------------------------------------
def shortcut_smooth(path, occ, n_iter=300):
    p = list(path)
    for _ in range(n_iter):
        if len(p) < 3:
            break
        i, j = sorted(rng.integers(0, len(p), 2))
        if j - i < 2:
            continue
        if collision_free(p[i], p[j], occ):
            p = p[:i + 1] + p[j:]
    return p

def path_length(path):
    return sum(hypot(b[0] - a[0], b[1] - a[1]) for a, b in zip(path, path[1:]))

# ---- Plotting ----------------------------------------------------------
def show_world(ax=None, title=''):
    if ax is None:
        _, ax = plt.subplots(figsize=(6, 6))
    for (a, b) in SEGMENTS:
        ax.plot([a[0], b[0]], [a[1], b[1]], 'k-', lw=2)
    ax.set_xlim(-0.3, 10.3); ax.set_ylim(-0.3, 10.3)
    ax.set_aspect('equal'); ax.set_title(title)
    return ax

def show_scan(pose, alphas, ranges, ax=None):
    ax = show_world(ax)
    x, y, th = pose
    xs = x + ranges * np.cos(alphas + th)
    ys = y + ranges * np.sin(alphas + th)
    for ex, ey in zip(xs, ys):
        ax.plot([x, ex], [y, ey], '-', color='#f39c12', lw=0.4, alpha=0.5)
    ax.plot(xs, ys, '.', ms=2.5, color='#d62828')
    ax.plot([x], [y], 'o', ms=10, color='#2f6ed3', zorder=5)
    return ax

def show_grid(grid_log, ax=None, title='', binary=False):
    if ax is None:
        _, ax = plt.subplots(figsize=(6, 6))
    if binary:
        ax.imshow(grid_log.T, origin='lower', cmap='gray_r',
                  vmin=0, vmax=1, extent=(0, 10, 0, 10))
    else:
        p = 1.0 / (1.0 + np.exp(-grid_log))
        ax.imshow(p.T, origin='lower', cmap='gray_r',
                  vmin=0, vmax=1, extent=(0, 10, 0, 10))
    ax.set_aspect('equal'); ax.set_title(title)
    return ax

def show_path(path, ax, color='#7b2cbf', label=None):
    xs = [p[0] for p in path]; ys = [p[1] for p in path]
    ax.plot(xs, ys, '-', color=color, lw=2.5, label=label)
    ax.plot(xs[0], ys[0], 'o', color='#2a9d3f', ms=10, zorder=6)
    ax.plot(xs[-1], ys[-1], '*', color='#2f6ed3', ms=15, zorder=6)

def show_tree(nodes, parents, ax, color='#888'):
    for i, p in enumerate(parents):
        if p is None:
            continue
        a, b = nodes[p], nodes[i]
        ax.plot([a[0], b[0]], [a[1], b[1]], '-', color=color, lw=0.5, alpha=0.7)

# ---- Grid <-> world conversion for planners ---------------------------
def grid_path_to_world(path_cells):
    return [((i + 0.5) * GRID_RES, (j + 0.5) * GRID_RES) for (i, j) in path_cells]

print('Setup ready. World:', len(SEGMENTS), 'segments.   Grid:',
      f'{GRID_M}x{GRID_N} cells at {GRID_RES} m/cell.')


Setup ready. World: 12 segments.   Grid: 100x100 cells at 0.1 m/cell.


## Part 1 — Sense (W10): a full 360° LiDAR scan

You're given `ray_segment_intersection(origin, direction, seg)` which
returns the ray-parameter `t ≥ 0` of the intersection or `None`.

**Write `scan(pose, n_beams=180)`** that sweeps `n_beams` bearings
uniformly over `[0, 2π)` and returns:

- `alphas`: array of bearings (radians, **in the robot frame**)
- `ranges`: array of distances (clipped at `max_range`)

For each bearing, cast a ray in **world** direction `alpha + theta`
from the pose, take the closest positive `t` over all `SEGMENTS`.


In [ ]:
def scan(pose, n_beams=180, max_range=10.0):
    """Return (alphas, ranges) for one full 360° scan.

    Args:
        pose: (x, y, theta) in world coordinates.
        n_beams: number of beams uniformly spread over [0, 2π).
        max_range: maximum sensor range.

    Hint: use ray_segment_intersection(origin, direction, seg).
    """
    x, y, theta = pose
    alphas = np.linspace(0, 2 * np.pi, n_beams, endpoint=False)
    ranges = np.full(n_beams, max_range)

    # TODO: for each alpha:
    #   1) world direction = (cos(alpha+theta), sin(alpha+theta))
    #   2) over SEGMENTS, find the smallest positive intersection distance
    #   3) clip at max_range and store in ranges[k]
    raise NotImplementedError("Part 1 — write the scan loop")

    return alphas, ranges


In [ ]:
pose = (2.0, 2.0, 0.0)
alphas, ranges = scan(pose, n_beams=180)
ax = show_scan(pose, alphas, ranges)
ax.set_title(f'Scan from {pose}  —  median range = {np.median(ranges):.2f} m')
plt.show()


## Part 2 — Map (W10): one log-odds occupancy update

A scan tells you *what's nearby right now*. To plan, you want a *map*
that fuses many scans. We use a log-odds grid:

- Cells the ray passed through are likely **free** → add `l_free` (negative).
- The cell where the ray *terminated* (only if the range came back short of
  `max_range`) is likely **occupied** → add `l_occ` (positive).
- Cells we never saw stay at `0` (unknown).

You're given `cells_along_ray(pose, alpha, r)` which returns
`(free_cells, hit_cell_or_None)`. Just apply the increments.


In [ ]:
def update_grid(grid, pose, alphas, ranges,
                l_free=-0.4, l_occ=0.85, l_clamp=5.0, max_range=10.0):
    """In-place log-odds update from one scan."""
    for a, r in zip(alphas, ranges):
        free, hit = cells_along_ray(pose, a, r, max_range)
        # TODO:
        #   - add l_free to every (i,j) in free
        #   - add l_occ to hit if hit is not None
        raise NotImplementedError("Part 2 — write the log-odds update")

    np.clip(grid, -l_clamp, l_clamp, out=grid)
    return grid


In [ ]:
# Drive 6 poses around the room, fuse them all into one map.
poses = [(1.5, 1.5, 0.5), (1.5, 8.5, -0.5), (8.5, 8.5, -2.5), (8.5, 1.5, 2.5),
         (5.0, 5.0, 0.0), (5.0, 5.0, np.pi)]

grid_log = np.zeros((GRID_M, GRID_N))
for p in poses:
    a, r = scan(p, n_beams=180)
    update_grid(grid_log, p, a, r)

binary = to_binary_occupancy(grid_log)
fig, axes = plt.subplots(1, 2, figsize=(13, 6))
show_grid(grid_log, axes[0], title='Fused log-odds map')
for (x, y, _) in poses:
    axes[0].plot(x, y, 'rx', ms=8)
show_grid(binary, axes[1], title='Binary occupancy (planner input)', binary=True)
plt.show()


## Part 3 — Plan deliberately (W11): A\*

Take the **inflated** binary map, pick a start and goal cell, and find
the shortest 8-connected path. You're given:

- `neighbors_8(i, j, M, N)` → yields `(ni, nj, step_cost)`
- `octile((i,j), (gi,gj))` → admissible heuristic
- `heapq` for the priority queue

A\* is Dijkstra with `f = g + h` instead of `f = g`.


In [ ]:
def astar(grid, start, goal):
    """Return (path, cost) on an 8-connected grid; path is a list of (i,j) cells.

    grid: 2-D uint8 array, 1 = blocked, 0 = free.
    """
    M, N = grid.shape
    INF = float('inf')
    g_cost = {start: 0.0}
    parent = {start: None}
    open_set = [(octile(start, goal), 0.0, start)]   # (f, g, cell)
    closed = set()

    # TODO: standard A* loop.
    #   while open_set:
    #     pop the cell with smallest f
    #     if it's the goal, reconstruct path via parent[] and return (path, g)
    #     skip if already closed
    #     for each (ni, nj, step) in neighbors_8(...) that's free:
    #         if g_cost[current] + step beats g_cost[neighbor], update parent,
    #         g_cost, and push (g + h, g, neighbor)
    raise NotImplementedError("Part 3 — write A*")


In [ ]:
inflated = inflate(binary, r_cells=3)
start_cell = world_to_grid(1.5, 1.5)
goal_cell  = world_to_grid(8.5, 8.5)
path_cells, cost = astar(inflated, start_cell, goal_cell)
print(f'A* cost = {cost:.2f} cells   |  {len(path_cells)} waypoints')

astar_path = grid_path_to_world(path_cells)
ax = show_grid(inflated, title='A* path on inflated map', binary=True)
show_path(astar_path, ax, color='#2f6ed3', label='A*')
ax.legend(loc='lower right'); plt.show()


## Part 4 — Plan with sampling (W12): RRT

Same map, same start and goal, but now in **continuous** world
coordinates. You're given `nearest`, `steer`, and `collision_free`.

**Write `rrt(start, goal, occ)`** that:

1. Initialises a tree with `start`.
2. Loops up to `max_iter` times: sample a random `q_rand` (with
   probability `goal_bias` sample the goal), find the nearest tree
   node, steer one step toward `q_rand`, and if the new edge is
   collision-free, append it as a child of the nearest node.
3. Stops once a node is within `goal_tol` of the goal and returns the
   path (start → … → goal).


In [ ]:
def rrt(start, goal, occ,
        max_iter=2000, step=0.5, goal_bias=0.1, goal_tol=0.4,
        bounds=(0.0, 10.0, 0.0, 10.0)):
    """Vanilla RRT in continuous 2-D. Returns (path_world, nodes, parents)."""
    nodes = [start]
    parents = [None]
    xmin, xmax, ymin, ymax = bounds

    for _ in range(max_iter):
        # TODO:
        #   1) sample q_rand: with prob goal_bias use `goal`, else uniform in bounds
        #   2) idx = nearest(nodes, q_rand)
        #   3) q_new = steer(nodes[idx], q_rand, step)
        #   4) if collision_free(nodes[idx], q_new, occ):
        #         append q_new and parent=idx
        #         if hypot(q_new - goal) < goal_tol -> reconstruct path and return
        raise NotImplementedError("Part 4 — write the RRT inner loop")

    return None, nodes, parents


In [ ]:
start_xy = (1.5, 1.5); goal_xy = (8.5, 8.5)
rrt_path, nodes, parents = rrt(start_xy, goal_xy, inflated)
print(f'RRT: tree size = {len(nodes)},  path length = {path_length(rrt_path):.2f} m')

ax = show_grid(inflated, title='RRT tree + path on inflated map', binary=True)
show_tree(nodes, parents, ax)
show_path(rrt_path, ax, color='#7b2cbf', label='RRT')
ax.legend(loc='lower right'); plt.show()


## Part 5 — Drive: smoothing makes the path executable

The raw RRT path is jagged (you saw it). Shortcut smoothing throws
random pairs of waypoints at line-of-sight checks; valid shortcuts
collapse the path. We also compare to the A\* path to see which one
the robot would actually prefer to drive.


In [ ]:
astar_smoothed = shortcut_smooth(astar_path, inflated)
rrt_smoothed  = shortcut_smooth(rrt_path,  inflated)

print(f'A*  raw {path_length(astar_path):5.2f} m  -> smoothed {path_length(astar_smoothed):5.2f} m')
print(f'RRT raw {path_length(rrt_path):5.2f} m  -> smoothed {path_length(rrt_smoothed):5.2f} m')

fig, axes = plt.subplots(1, 2, figsize=(13, 6))
show_grid(inflated, axes[0], title='A* (raw vs smoothed)', binary=True)
show_path(astar_path,     axes[0], color='#bbbbbb', label='raw')
show_path(astar_smoothed, axes[0], color='#2f6ed3', label='smoothed')
axes[0].legend(loc='lower right')

show_grid(inflated, axes[1], title='RRT (raw vs smoothed)', binary=True)
show_path(rrt_path,     axes[1], color='#bbbbbb', label='raw')
show_path(rrt_smoothed, axes[1], color='#7b2cbf', label='smoothed')
axes[1].legend(loc='lower right')
plt.show()


## Wrap-up

You just built the full perception → planning → action pipeline of a
mobile robot in **about 60 lines** of your own code:

- `scan` was the simulated LiDAR.
- `update_grid` fused readings into a probabilistic map.
- `astar` and `rrt` produced two complementary plans: the first
  optimal-on-grid, the second flexible-in-continuous-space.
- Smoothing turned both plans into something a real controller could
  execute.

**Things worth noticing:**

- A\* and RRT solve the *same* problem on the *same* map but use very
  different machinery — and they pay different costs.
  Compare their path lengths and runtimes above.
- The map quality (Part 2) bounds everything that follows. Bad map
  → bad plan, no matter how clever the planner.
- Inflating the obstacles by the robot radius is what lets us treat
  the robot as a point in Parts 3 and 4. Without it, A\* and RRT
  would happily plan a path that grazes a wall.

**Where to go next:** RRT\* (rewire neighbors so the cost-to-come
shrinks as the tree grows), bidirectional RRT (grow trees from start
*and* goal), and replacing the LiDAR with a real ROS topic.
